# M1-B1 — Modélisation de l'historique des réservations

## Tâche 4 — Entraînement du modèle V1
| Jeu | Nombre de réservations |
|---|---:|
| Entraînement | 58 792 |
| Test | 19 911 |

Les metriques sur le jeu de test:
| Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---:|---:|---:|---:|---:|
| 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |

**Ces performances parfaites sont le signe d'une fuite de données.**

#### Les variables suivantes sont à l'origine de cette fuite de données:

| Variable | Problème potentiel |
|---|---|
| `reservation_status` | Révèle directement si la réservation a été annulée. |
| `reservation_status_date` | Peut révéler la date et le moment de l’annulation. |
| `reservation_id` | Peut introduire des informations liées à l’ordre de création des réservations. |
| `assigned_room_type` | Peut contenir une information connue après la réservation. |
| `arrival_date` | À examiner selon le moment où la prédiction est effectuée. |


#### Décision: 
*Supprimer les variables problématiques et entraîner un nouveau modèle (cancellation_model_v1.joblib)*

### Evaluation du modèle cancellation_model_v1.joblib
Nombre d'échantillons : **78703**

Les métriques sur le jeu de test :

| Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---:|---:|---:|---:|---:|
| 0.8850 | 0.8647 | 0.8088 | 0.8358 | 0.9516 |

#### Interprétation :
- Accuracy (88,50 %) : le modèle classe correctement environ 89 % des réservations.
- Precision (86,47 %) : parmi les réservations prédites comme annulées, environ 86 % le sont réellement.
- Recall (80,88 %) : le modèle détecte environ 81 % des annulations réelles.
- F1-score (83,58 %) : bon compromis entre précision et rappel.
- ROC-AUC (95,16 %) : le modèle distingue très bien les réservations annulées des réservations non annulées.

### À quel moment chaque variable est-elle connue ?
Prenons le cycle de vie d'une réservation hôtelière.

#### 1. Au moment de la réservation

Informations disponibles dès la création :
- hotel
- lead_time
- arrival_date_week_number
- arrival_date_day_of_month
- adults, children, babies
- meal
- country
- market_segment
- distribution_channel
- is_repeated_guest
- previous_cancellations
- previous_bookings_not_canceled
- customer_type
- deposit_type

#### 2. Après la création de la réservation

Informations qui peuvent évoluer :
- booking_changes
- agent
- company
- assigned_room_type
- reserved_room_type
- total_of_special_requests

Attention : certaines sont disponibles dès la réservation, d'autres seulement après une modification ou une affectation.

#### 3. À l'approche du séjour

Informations qui deviennent progressivement disponibles :
- reservation_status
- reservation_status_date

Elles décrivent l'état de la réservation et son évolution.

#### 4. Après le séjour ou la clôture de la réservation

Informations potentiellement postérieures :
- reservation_status_Check-Out
- reservation_status_Canceled
- reservation_status_No-Show

Elles indiquent comment la réservation s'est terminée.

## Tâche 5 — Validations du modèle V1

Nombre d'échantillons : **78703**
Jeu de test : **15 741 échantillons**
### Validation avec StratifiedKFold

#### StratifiedKFold — résultats par fold (test_f1 = validation) :
| Fold | Accuracy | Precision | Recall | Train F1 | Validation F1 | Écart train-validation | ROC-AUC |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 0 | 0.882981 | 0.870085 | 0.795392 | 0.857891 | 0.831063 | 0.026827 | 0.948541 |
| 1 | 0.885882 | 0.876575 | 0.796813 | 0.857349 | 0.834793 | 0.022556 | 0.951340 |
| 2 | 0.881927 | 0.873867 | 0.787332 | 0.857300 | 0.828346 | 0.028955 | 0.948203 |
#### StratifiedKFold — moyenne et écart-type par métrique :

| Métrique | Moyenne | Écart-type |
|---|---:|---:|
| Accuracy | 0.883596 | 0.002048 |
| Precision | 0.873509 | 0.003260 |
| Recall | 0.793179 | 0.005113 |
| Train F1 | 0.857513 | 0.000328 |
| Validation F1 | 0.831401 | 0.003237 |
| Écart train-validation | 0.026113 | 0.003259 |
| ROC-AUC | 0.949361 | 0.001722 |


 **La moyenne du F1 vaut environ 85.75 % à l’entraînement contre 83.14 % en validation. L’écart moyen est de 2.61 points : il est faible, donc le surapprentissage ne semble pas important.**

### Résultats de GridSearchCV :
   | max_iter | learning_rate | F1 moyen | Écart-type F1 |
   |---:|---:|---:|---:|
   | 100 | 0.05 | 0.814213 | 0.003014 |
   | 200 | 0.05 | 0.823255 | 0.004180 |
   | 100 | 0.10 | 0.822909 | 0.002935 |
   | 200 | 0.10 | 0.831401 | 0.002643 |

#### Conclusion
Meilleur résultat parmi ces essais : 

**max_iter=200, learning_rate=0,1, F1 moyen 0,8314 (écart-type 0,0026).**

#### Validation avec TimeSeriesSplit

#### TimeSeriesSplit — résultats par fold (test_f1 = validation) :
| Fold | Accuracy | Precision | Recall | Train F1 | Validation F1 | Écart train-validation | ROC-AUC |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 0 | 0.850853 | 0.733055 | 0.756631 | 0.932063 | 0.744657 | 0.187407 | 0.904222 |
| 1 | 0.804346 | 0.829421 | 0.567300 | 0.931992 | 0.673765 | 0.258228 | 0.870407 |
| 2 | 0.827981 | 0.846207 | 0.644950 | 0.901569 | 0.731997 | 0.169572 | 0.903945 |
| 3 | 0.762794 | 0.876111 | 0.410356 | 0.885995 | 0.558923 | 0.327072 | 0.855953 |
| 4 | 0.774421 | 0.889598 | 0.458407 | 0.865625 | 0.605039 | 0.260586 | 0.862298 |

#### TimeSeriesSplit — moyenne et écart-type par métrique :
| Métrique | Moyenne | Écart-type |
|---|---:|---:|
| Accuracy | 0.804079 | 0.036549 |
| Precision | 0.834878 | 0.061686 |
| Recall | 0.567529 | 0.139968 |
| Train F1 | 0.903449 | 0.029036 |
| Validation F1 | 0.662876 | 0.080212 |
| Écart train-validation | 0.240573 | 0.063368 |
| ROC-AUC | 0.879365 | 0.023139 |

#### Comparaison des deux types de validations du modèle v1

| Validation | F1 entraînement moyen | F1 validation moyen | Écart |
|---|---:|---:|---:|
| Stratifiée, 3 folds | 0,8575 | 0,8314 | 0,0261 |
| Temporelle, 5 folds | 0,9034 | 0,6629 | 0,2406 |

### Conclusion:

Le F1 de validation baisse donc d’environ 16,9 points avec TimeSeriesSplit. La validation stratifiée mélange les dates entre folds, tandis que la validation temporelle entraîne sur des périodes antérieures et valide sur des périodes ultérieures. Elle révèle ainsi davantage les variations dans le temps et la difficulté à généraliser aux réservations plus récentes. Le script s’exécute avec succès.